In [ ]:
!pip install jsonlines

In [ ]:
import pandas as pd
from pathlib import Path


# Repo root, resolved relative to this file — works regardless of where you run it from.
ROOT = Path(__file__).resolve().parents[1]   # notebook: Path.cwd().parents[0]
DATA = ROOT / "01_Knowledge base"

df = pd.read_csv(DATA / "KnowledgeBase_Rev02.csv")

# Display the first few rows of the DataFrame to verify
print(df.head())

In [ ]:
import json

folder_path = "pertanyaan_json"

def read_pertanyaan_user(folder_path):
    rows = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".jsonl"):
            file_path = os.path.join(folder_path, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        data = json.loads(line)
                        data["source_file"] = filename  # optional: track file name
                        rows.append(data)

    df = pd.DataFrame(rows)

    return df


In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# model_transformers = "sentence-transformers/all-MiniLM-L6-v2"
# model_transformers = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_transformers = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
transformers_model_similarity = SentenceTransformer(model_transformers)

def transformers_get_context(question, k_value):
  # Get embeddings
  filtered_embedding = transformers_model_similarity.encode(df['Pertanyaan'].astype(str).tolist(),convert_to_tensor=True)
  query_embedding = transformers_model_similarity.encode(question, convert_to_tensor=True)

  # Compute cosine similarities
  cos_scores = util.cos_sim(query_embedding, filtered_embedding)[0]

  # Get top-k score and index
  top_score, top_idx = torch.topk(cos_scores, k=k_value)

  # print("Top scores:", top_score)
  # print("Top indexes:", top_idx)

  fallback_pertanyaan = "Kapan sih Prodi Informatika ini mulai berdiri?"

  results = []

  for score, idx in zip(top_score, top_idx):
    score_value = score.item()
    idx_value = idx.item()

    if score_value * 100 > 50:
      results.append((
          df.iloc[idx_value]["Pertanyaan"],
          score_value,
          df.iloc[idx_value]["Jawaban"]
      ))

  # If at least one result above 50%, return matched results
  if len(results) > 0:
    return results

  # If all top-k results below 50%, return fallback once
  fallback_row = df[df["Pertanyaan"] == fallback_pertanyaan].iloc[0]

  return [(
      fallback_row["Pertanyaan"],
      top_score[0].item(),
      fallback_row["Jawaban"]
  )]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

new_max_token = 256
chat_model_id = "aisingapore/Llama-SEA-LION-v3.5-8B-R"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_id)

chat_model = AutoModelForCausalLM.from_pretrained(
                  chat_model_id,
                  dtype=torch.bfloat16
              ).to(DEVICE)

def chatbot_respond(message):
  input_ids = chat_tokenizer.apply_chat_template(
      message,
      add_generation_prompt=True,
      tokenize=False,
      thinking_mode="off"
  )

  inputs = chat_tokenizer(input_ids, return_tensors="pt")

  input_id = inputs["input_ids"].to(DEVICE)
  attention_mask=inputs["attention_mask"].to(DEVICE)

  outputs = chat_model.generate(
          input_id,
          attention_mask=attention_mask,
          max_new_tokens=new_max_token,
          pad_token_id=chat_tokenizer.eos_token_id,
          temperature=0.3,
          do_sample=True,
          repetition_penalty=1.1
      )

  answer = chat_tokenizer.decode(outputs[0], skip_special_tokens=True)

  return {"choices": [{"message": {"role": "assistant", "content": answer}}]}

In [ ]:
import jsonlines
import os
from datetime import date, datetime

def save_json(k_value, save_dir, question, similar_question, similar_score,
              model_similarity, targeted_response, model_response, complete_response, time_response, username):

    save_file = os.path.join(save_dir, str(date.today())+ '_llm_rag_only_top_k_'+ str(k_value) + '.jsonl')
    os.makedirs(os.path.dirname(save_file), exist_ok=True)
    now = datetime.now()
    time = now.strftime("%Y:%m:%d %H:%M:%S")

    with jsonlines.open(save_file, mode='a') as writer:
        writer.write(
        {
            "time": time,
            "question": question,
            "similar_question": similar_question,
            "similar_score": similar_score,
            "model_similarity": model_similarity,
            "targeted_response": targeted_response,
            "model_response": model_response,
            "complete_response": complete_response,
            "time_response": str(time_response) + " seconds",
            "username": username
        }
    )

In [ ]:
from tqdm import tqdm

def main():
    all_questions = read_pertanyaan_user(folder_path)["question"].astype(str).tolist()
    k_values = [1]
    scenario = "llm_rag"    #llm, llm_rag, llm_rag_prompt_adapt
    for k_value in k_values:
        for user_message in tqdm(all_questions):
            response = {}
            sim_questions = []
            sim_scores = []
            descriptions = []
            start_time = datetime.now()

            if scenario == "llm_rag_prompt_adapt":
                result_context = transformers_get_context(user_message, k_value)
                for (sim_question, sim_score, description) in result_context:
                    sim_questions.append(sim_question)
                    sim_scores.append(sim_score)
                    descriptions.append(description)
                description_detailed = " ".join(descriptions)

                content = f"Ini adalah jawaban anda tentang deskripsi \
                \"{description_detailed}\". Saya adalah pengunjung pada Program Studi Informatika kampus UHN I Gusti Bagus Sugriwa \
                ini adalah pertanyaan saya: \"{user_message}\"? Buatlah pendapat yang singkat, sopan dan bersahabat \
                untuk saya dengan gaya bahasa berusia 18 tahun dan jangan memberikan pertanyaan lanjutan. Jawablah dalam 50 kata."

            elif scenario == "llm":
                description_detailed = ""
                content = user_message

            elif scenario == "llm_rag":
                result_context = transformers_get_context(user_message, k_value)
                for (sim_question, sim_score, description) in result_context:
                    sim_questions.append(sim_question)
                    sim_scores.append(sim_score)
                    descriptions.append(description)
                description_detailed = " ".join(descriptions)

                content = f"Ini adalah jawaban anda tentang deskripsi \
                \"{description_detailed}\". Ini adalah pertanyaan saya: \"{user_message}\"?"


            # print (content)
            # print (sim_questions)
            # print (sim_scores)
            messages = [
                    {"role": "user", "content": content},
                ]

            bot_response = chatbot_respond(messages)
            response["text"] = bot_response['choices'][0]['message']['content'].split("assistant\n\n")[-1].replace("<think>\n\n</think>\n\n", "")

            end_time = datetime.now()
            time_difference = end_time - start_time
            seconds = int(time_difference.total_seconds())
            try:
                save_json(
                    k_value=k_value,
                    save_dir=save_dir,
                    question=user_message,
                    similar_question=sim_questions,
                    similar_score=sim_scores,
                    model_similarity=model_transformers,
                    targeted_response=description_detailed,
                    model_response=response["text"],
                    complete_response=bot_response,
                    time_response=seconds,
                    username="generated")
            except:
                print("cannot save json files")

In [ ]:
main()